In [ ]:
# @title Default title text
#!/usr/bin/env python3
"""
SMILES to PDBQT Converter with AutoDock Vina Preparation
Designed for Google Colab with Google Drive integration,
including result zipping, download functionality,
PDB and PDBQT files organized into separate subfolders,
and a 'ligands.txt' file listing all generated PDBQT files.
"""
!pip install pandas numpy rdkit py3Dmol
# Standard library imports
import os
import sys
import subprocess
import zipfile
import urllib.request
import time # Used for simulating processing time and timestamping reports

# Third-party library imports
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors
from rdkit.Chem import rdForceFieldHelpers
import py3Dmol
from IPython.display import display, HTML
import warnings

# Google Colab specific imports
try:
    from google.colab import drive
    from google.colab import files
except ImportError:
    print("Warning: 'google.colab' modules not found. Google Drive mounting and file download functionality will be skipped.")
    drive = None
    files = None

# Suppress RDKit warnings for cleaner output
warnings.filterwarnings('ignore')

# --- Google Drive Setup ---
work_dir = os.getcwd() # Default to current directory
if drive:
    print("Mounting Google Drive...")
    try:
        drive.mount('/content/drive')
        # Set working directory in Google Drive
        work_dir = "/content/drive/MyDrive/Ligand_Processing"
        os.makedirs(work_dir, exist_ok=True)
        os.chdir(work_dir)
        print(f"Working directory set to: {work_dir}")
    except Exception as e:
        print(f"Error mounting Google Drive or setting working directory: {e}")
        print("Proceeding with default Colab working directory.")
else:
    print("Google Colab environment not detected. Using current working directory.")


class LigandProcessor:
    def __init__(self, base_output_dir="processed_ligands"):
        self.molecules = []
        self.original_names = []  # Store original compound names
        self.force_field_options = {
            'MMFF94': 'MMFF94 - Merck Molecular Force Field',
            'MMFF94s': 'MMFF94s - MMFF94 static variant',
            'UFF': 'UFF - Universal Force Field',
            'ETKDG': 'ETKDG - Experimental-Torsion-angle Knowledge Distance Geometry'
        }
        self.selected_force_field = None
        self.obabel_path = None

        # Define output directories
        self.base_output_dir = base_output_dir
        self.pdb_output_dir = os.path.join(self.base_output_dir, "PDB_files")
        self.pdbqt_output_dir = os.path.join(self.base_output_dir, "PDBQT_files")
        self.report_output_dir = self.base_output_dir # Report stays in base output

        # Create output directories if they don't exist
        os.makedirs(self.pdb_output_dir, exist_ok=True)
        os.makedirs(self.pdbqt_output_dir, exist_ok=True)
        os.makedirs(self.report_output_dir, exist_ok=True) # Ensure base exists for report

        print(f"LigandProcessor initialized. Base output directory: {self.base_output_dir}")
        print(f"  PDB files will be saved in: {self.pdb_output_dir}")
        print(f"  PDBQT files will be saved in: {self.pdbqt_output_dir}")

    def install_dependencies(self):
        """Install required dependencies for Google Colab"""
        print("Installing dependencies...")

        try:
            # Method 1: Try apt-get first (faster and more reliable in Colab)
            print("Installing OpenBabel via apt...")
            subprocess.run(['apt-get', 'update', '-qq'], check=True, capture_output=True)
            subprocess.run(['apt-get', 'install', '-y', 'openbabel'], check=True, capture_output=True)
            print("✓ OpenBabel installed via apt")

        except subprocess.CalledProcessError as e:
            print(f"Apt installation failed: {e.stderr.strip()}")
            try:
                # Method 2: Try pip installation
                print("Installing OpenBabel and related tools via pip...")
                subprocess.run(['pip', 'install', 'openbabel-wheel'], check=True, capture_output=True)
                subprocess.run(['pip', 'install', 'meeko'], check=True, capture_output=True)
                print("✓ OpenBabel installed via pip")

            except subprocess.CalledProcessError as e:
                print(f"Pip installation failed: {e.stderr.strip()}")
                try:
                    # Method 3: Install conda/mamba as backup
                    print("Installing via conda...")
                    subprocess.run(['wget', '-q', 'https://github.com/conda-forge/miniforge/releases/latest/download/Mambaforge-Linux-x86_64.sh'], check=False, capture_output=True)
                    subprocess.run(['bash', 'Mambaforge-Linux-x86_64.sh', '-b', '-p', '/opt/conda'], check=False, capture_output=True)
                    os.environ['PATH'] = '/opt/conda/bin:' + os.environ['PATH']

                    subprocess.run(['/opt/conda/bin/mamba', 'install', '-c', 'conda-forge', 'openbabel', '-y'], check=False, capture_output=True)
                    subprocess.run(['/opt/conda/bin/mamba', 'install', '-c', 'conda-forge', 'meeko', '-y'], check=False, capture_output=True)
                    print("✓ Dependencies installed via conda")

                except Exception as e:
                    print(f"⚠ Could not install OpenBabel automatically via any method: {e}")
                    print("Will use manual PDBQT conversion method if OpenBabel is not found.")

        # Verify installation
        obabel_paths = ['/usr/bin/obabel', '/opt/conda/bin/obabel', 'obabel']
        self.obabel_path = None

        for path in obabel_paths:
            try:
                # Check if obabel command exists and is executable
                if os.path.exists(path) and os.access(path, os.X_OK):
                    result = subprocess.run([path, '--help'], capture_output=True, text=True, timeout=5)
                    if result.returncode == 0:
                        self.obabel_path = path
                        print(f"✓ OpenBabel found at: {path}")
                        break
            except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                continue

        if not self.obabel_path:
            print("⚠ OpenBabel not found - will use manual PDBQT generation (simplified charges/types)")

        print("Dependency installation completed!")

    def load_smiles(self, smiles_input):
        """Load SMILES from various input formats (string, list, CSV, TXT)"""
        self.molecules = []
        self.original_names = []
        smiles_list = []

        if isinstance(smiles_input, str):
            # Single SMILES string or file path
            if os.path.exists(smiles_input):
                print(f"Loading file: {smiles_input}")
                # File input
                if smiles_input.lower().endswith('.csv'):
                    try:
                        df = pd.read_csv(smiles_input)
                        print(f"CSV loaded successfully. Shape: {df.shape}")
                        print("Available columns:", df.columns.tolist())

                        # Auto-detect SMILES column
                        smiles_column = None
                        possible_smiles_names = ['smiles', 'smi', 'canonical_smiles', 'smiles_string', 'mol', 'molecule']

                        for col in df.columns:
                            if col.lower() in possible_smiles_names:
                                smiles_column = col
                                print(f"Auto-detected SMILES column: '{col}'")
                                break

                        if not smiles_column:
                            print("\nCould not auto-detect SMILES column.")
                            print("Please specify which column contains SMILES:")
                            for i, col in enumerate(df.columns):
                                print(f"{i+1}. {col}")

                            while True:
                                try:
                                    col_choice = input("Enter column number or name: ").strip()
                                    if col_choice.isdigit():
                                        col_idx = int(col_choice) - 1
                                        if 0 <= col_idx < len(df.columns):
                                            smiles_column = df.columns[col_idx]
                                            break
                                    elif col_choice in df.columns:
                                        smiles_column = col_choice
                                        break
                                    else:
                                        print("Invalid selection. Please try again.")
                                except (ValueError, IndexError):
                                    print("Invalid selection. Please try again.")

                        # Look for compound name column
                        name_column = None
                        possible_name_cols = ['compound_name', 'name', 'id', 'compound_id', 'mol_name', 'title', 'compound']

                        for col in df.columns:
                            if col.lower() in possible_name_cols:
                                name_column = col
                                print(f"Found compound name column: '{col}'")
                                break

                        smiles_list = df[smiles_column].dropna().astype(str).tolist()

                        # Get compound names if available, otherwise generate generic names
                        if name_column:
                            self.original_names = df[name_column].astype(str).tolist()
                            # Ensure original_names list matches length of smiles_list
                            if len(self.original_names) > len(smiles_list):
                                self.original_names = self.original_names[:len(smiles_list)]
                            elif len(self.original_names) < len(smiles_list):
                                # Pad with generic names if names are missing for some SMILES
                                self.original_names.extend([f"compound_{i+1}" for i in range(len(self.original_names), len(smiles_list))])
                        else:
                            print("No compound name column found - will use generic names.")
                            self.original_names = [f"compound_{i+1}" for i in range(len(smiles_list))]

                        print(f"Found {len(smiles_list)} SMILES in column '{smiles_column}'")

                        # Display first few SMILES for verification
                        print("First 3 entries:")
                        for i in range(min(3, len(smiles_list))):
                            name = self.original_names[i] if i < len(self.original_names) else f"compound_{i+1}"
                            print(f"  {i+1}. {name}: {smiles_list[i]}")

                    except Exception as e:
                        print(f"Error reading CSV file: {e}")
                        return 0
                elif smiles_input.lower().endswith(('.txt', '.smi')):
                    # Text file with one SMILES per line
                    with open(smiles_input, 'r') as f:
                        smiles_list = [line.strip() for line in f if line.strip()]
                    self.original_names = [f"compound_{i+1}" for i in range(len(smiles_list))]
                else:
                    # Single SMILES string
                    smiles_list = [smiles_input]
                    self.original_names = ["ligand_1"]
        elif isinstance(smiles_input, list):
            # List of SMILES
            smiles_list = smiles_input
            self.original_names = [f"ligand_{i+1}" for i in range(len(smiles_list))]

        # Convert SMILES to RDKit molecules
        successful_conversions = 0
        for i, smi in enumerate(smiles_list):
            try:
                mol = Chem.MolFromSmiles(str(smi).strip())
                if mol is not None:
                    # Sanitize compound name for filename use
                    compound_name = str(self.original_names[i]).replace(' ', '_').replace('/', '_').replace('\\', '_')
                    compound_name = ''.join(c for c in compound_name if c.isalnum() or c in '_-.')

                    mol.SetProp("_Name", compound_name)
                    mol.SetProp("SMILES", str(smi).strip())
                    mol.SetProp("OriginalName", str(self.original_names[i])) # Keep original for report

                    self.molecules.append(mol)
                    successful_conversions += 1
                else:
                    print(f"Warning: Could not parse SMILES {i+1} ('{smi}'). Skipping.")
            except Exception as e:
                print(f"Error processing SMILES {i+1} ('{smi}'): {e}. Skipping.")

        print(f"Successfully loaded {successful_conversions} molecules out of {len(smiles_list)} SMILES")
        return successful_conversions

    def select_force_field(self):
        """Interactive force field selection"""
        print("\nAvailable Force Fields:")
        print("=" * 50)
        for key, description in self.force_field_options.items():
            print(f"{key}: {description}")
        print("=" * 50)

        while True:
            choice = input("Select force field (MMFF94/MMFF94s/UFF/ETKDG): ").strip().upper()
            if choice in self.force_field_options.keys():
                self.selected_force_field = choice
                print(f"Selected force field: {self.force_field_options[choice]}")
                break
            else:
                print("Invalid choice. Please select from the available options.")

    def generate_3d_conformers(self, mol, force_field):
        """Generate 3D conformers using selected force field"""
        # Add hydrogens
        mol_h = Chem.AddHs(mol)

        if force_field == 'ETKDG':
            # ETKDG method
            params = AllChem.ETKDGv3()
            params.randomSeed = 42
            AllChem.EmbedMolecule(mol_h, params)
            # ETKDG usually benefits from a post-optimization with a classical force field
            try:
                AllChem.MMFFOptimizeMolecule(mol_h)
            except ValueError: # MMFF might fail for some molecules
                print("Warning: MMFF optimization failed for ETKDG conformer. Skipping optimization.")
        else:
            # Embed molecule
            AllChem.EmbedMolecule(mol_h, randomSeed=42)

            # Optimize with selected force field
            try:
                if force_field == 'MMFF94':
                    AllChem.MMFFOptimizeMolecule(mol_h)
                elif force_field == 'MMFF94s':
                    AllChem.MMFFOptimizeMolecule(mol_h, mmffVariant='MMFF94s')
                elif force_field == 'UFF':
                    AllChem.UFFOptimizeMolecule(mol_h)
            except ValueError:
                print(f"Warning: {force_field} optimization failed. Skipping optimization.")

        return mol_h

    def convert_to_pdb(self):
        """Convert molecules to PDB format and save in PDB_files folder"""
        if not self.selected_force_field:
            self.select_force_field()

        pdb_files = []
        print(f"\nGenerating 3D conformers using {self.selected_force_field}...")

        for i, mol in enumerate(self.molecules):
            try:
                # Generate 3D conformer
                mol_3d = self.generate_3d_conformers(mol, self.selected_force_field)

                # Save as PDB in the dedicated PDB folder
                mol_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"ligand_{i+1}"
                pdb_filename = os.path.join(self.pdb_output_dir, f"{mol_name}.pdb")

                # Write PDB file
                pdb_block = Chem.MolToPDBBlock(mol_3d)
                with open(pdb_filename, 'w') as f:
                    f.write(pdb_block)

                pdb_files.append(pdb_filename)
                print(f"Created: {pdb_filename}")

            except Exception as e:
                print(f"Error processing molecule '{mol.GetProp('OriginalName') if mol.HasProp('OriginalName') else f'ligand_{i+1}'}': {str(e)}. Skipping PDB generation.")

        print(f"\nSuccessfully created {len(pdb_files)} PDB files")
        return pdb_files

    def prepare_autodock_vina(self, pdb_files):
        """Prepare ligands for AutoDock Vina (Method 1: Traditional ADT-like approach)"""
        print("\nPreparing ligands for AutoDock Vina...")
        pdbqt_files = []

        for pdb_file in pdb_files:
            try:
                # Load molecule from PDB
                mol = Chem.MolFromPDBFile(pdb_file, removeHs=False)
                if mol is None:
                    print(f"Warning: Could not load {pdb_file}. Skipping PDBQT conversion.")
                    continue

                print(f"\nProcessing {os.path.basename(pdb_file)}...")

                # Step 1: Initial Processing (atoms are formatted by RDKit PDB export)
                print("✓ Step 1: Initial processing completed")

                # Step 2: Hydrogen atoms (already added during 3D generation)
                print("✓ Step 2: Polar hydrogens already added")

                # Step 3: Assign partial charges (Gasteiger charges for ligands)
                AllChem.ComputeGasteigerCharges(mol)
                print("✓ Step 3: Gasteiger charges assigned")

                # Step 4: Rotatable bonds identification
                num_rotatable_bonds = rdMolDescriptors.CalcNumRotatableBonds(mol)
                print(f"✓ Step 4: Identified {num_rotatable_bonds} rotatable bonds")

                # Step 5: Convert to PDBQT and save in PDBQT_files folder
                mol_name = os.path.basename(pdb_file).replace('.pdb', '')
                pdbqt_filename = os.path.join(self.pdbqt_output_dir, f"{mol_name}.pdbqt")

                # Try OpenBabel first if available
                if self.obabel_path:
                    try:
                        # Using --addpolarh for explicit polar hydrogens, --partialcharge for Gasteiger
                        cmd = f"{self.obabel_path} {pdb_file} -O {pdbqt_filename} --addpolarh --partialcharge gasteiger"
                        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, check=True)

                        if os.path.exists(pdbqt_filename) and os.path.getsize(pdbqt_filename) > 0:
                            print(f"✓ Step 5: Successfully created {os.path.basename(pdbqt_filename)} using OpenBabel")
                            pdbqt_files.append(pdbqt_filename)
                            self.verify_pdbqt(pdbqt_filename)
                            continue
                        else:
                            print(f"⚠ OpenBabel command succeeded but output file is empty or not found. Stderr: {result.stderr.strip()}")
                            print("Falling back to manual PDBQT generation...")
                    except subprocess.CalledProcessError as e:
                        print(f"⚠ OpenBabel failed with error: {e.stderr.strip()}")
                        print("Falling back to manual PDBQT generation...")
                    except Exception as e:
                        print(f"⚠ OpenBabel error: {e}")
                        print("Falling back to manual PDBQT generation...")

                # Manual PDBQT generation (fallback method)
                success = self.manual_pdbqt_conversion(pdb_file, pdbqt_filename, num_rotatable_bonds, mol)
                if success:
                    print(f"✓ Step 5: Successfully created {os.path.basename(pdbqt_filename)} using manual method")
                    pdbqt_files.append(pdbqt_filename)
                    self.verify_pdbqt(pdbqt_filename)
                else:
                    print(f"✗ Failed to create PDBQT for {os.path.basename(pdb_file)}")

            except Exception as e:
                print(f"Error preparing {os.path.basename(pdb_file)} for AutoDock Vina: {str(e)}. Skipping.")

        print(f"\nSuccessfully prepared {len(pdbqt_files)} PDBQT files")
        return pdbqt_files

    def manual_pdbqt_conversion(self, pdb_file, pdbqt_filename, num_rotatable_bonds, mol):
        """
        Manual conversion from PDB to PDBQT format.
        This method is a fallback and provides simplified atom types and charges.
        For accurate charges and types, OpenBabel or Meeko is highly recommended.
        """
        try:
            print("  Using manual PDBQT conversion (simplified charges/types)...")

            with open(pdb_file, 'r') as f:
                pdb_lines = f.readlines()

            pdbqt_lines = []

            # Add PDBQT header
            mol_name = mol.GetProp("_Name") if mol.HasProp("_Name") else os.path.basename(pdb_file).replace('.pdb', '')
            pdbqt_lines.append(f"REMARK  Name = {mol_name}\n")
            pdbqt_lines.append(f"REMARK  {num_rotatable_bonds} active torsions:\n")
            pdbqt_lines.append("REMARK  status: ('A' for Active; 'I' for Inactive)\n")
            pdbqt_lines.append("ROOT\n")

            # Convert PDB atoms to PDBQT format
            for line in pdb_lines:
                if line.startswith(('ATOM', 'HETATM')):
                    # Parse PDB line
                    atom_serial = int(line[6:11])
                    atom_name = line[12:16].strip()
                    res_name = line[17:20].strip()
                    chain_id = line[21:22].strip()
                    res_seq = int(line[22:26])
                    x = float(line[30:38])
                    y = float(line[38:46])
                    z = float(line[46:54])
                    occupancy = float(line[54:60])
                    temp_factor = float(line[60:66])
                    element_symbol = line[76:78].strip() # Get element from PDB column 77-78

                    # Assign AutoDock atom types
                    autodock_type = self.get_autodock_type(element_symbol)

                    # Get partial charge from RDKit's Gasteiger charges
                    partial_charge = 0.0
                    for atom in mol.GetAtoms():
                        # A more robust solution would map PDB atom serials to RDKit atom indices
                        # For simplicity, we'll try to match by element and assume order
                        if atom.GetSymbol() == element_symbol and hasattr(atom, 'GetProp') and atom.HasProp('_GasteigerCharge'):
                            partial_charge = float(atom.GetProp('_GasteigerCharge'))
                            break # Found charge for this element, move to next PDB line

                    # Format PDBQT line
                    # PDBQT format: ATOM   6  N   ALA A   1      20.617  22.285  15.006  1.00  0.00   -0.300 N
                    pdbqt_line = (
                        f"ATOM  {atom_serial:5d} {atom_name:<4} {res_name} {chain_id}{res_seq:4d}    "
                        f"{x:8.3f}{y:8.3f}{z:8.3f}{occupancy:6.2f}{temp_factor:6.2f}    "
                        f"{partial_charge:+6.3f} {autodock_type:<2}\n"
                    )
                    pdbqt_lines.append(pdbqt_line)

            # Add PDBQT footer
            pdbqt_lines.append("ENDROOT\n")
            pdbqt_lines.append(f"TORSDOF {num_rotatable_bonds}\n")

            # Write PDBQT file
            with open(pdbqt_filename, 'w') as f:
                f.writelines(pdbqt_lines)

            return os.path.exists(pdbqt_filename) and os.path.getsize(pdbqt_filename) > 0

        except Exception as e:
            print(f"  Manual conversion failed for {os.path.basename(pdb_file)}: {e}")
            return False

    def get_autodock_type(self, element_symbol):
        """Get AutoDock atom type from element symbol (simplified)"""
        autodock_type_map = {
            'C': 'C', 'N': 'N', 'O': 'O', 'S': 'S', 'P': 'P', 'F': 'F',
            'Cl': 'Cl', 'Br': 'Br', 'I': 'I', 'H': 'H',
        }
        return autodock_type_map.get(element_symbol, element_symbol) # Fallback to element symbol

    def verify_pdbqt(self, pdbqt_file):
        """Verify PDBQT file quality"""
        try:
            with open(pdbqt_file, 'r') as f:
                content = f.read()

            # Check for essential PDBQT components
            has_root = 'ROOT' in content
            has_endroot = 'ENDROOT' in content
            has_torsdof = 'TORSDOF' in content
            has_atoms = any(line.startswith('ATOM') or line.startswith('HETATM') for line in content.split('\n'))

            print(f"  PDBQT Verification for {os.path.basename(pdbqt_file)}:")
            print(f"    ✓ Has atoms: {has_atoms}")
            print(f"    ✓ Has ROOT section: {has_root}")
            print(f"    ✓ Has ENDROOT: {has_endroot}")
            print(f"    ✓ Has TORSDOF: {has_torsdof}")

            if all([has_atoms, has_root, has_endroot, has_torsdof]):
                print(f"    ✓ {os.path.basename(pdbqt_file)} is ready for AutoDock Vina!")
            else:
                print(f"    ⚠ {os.path.basename(pdbqt_file)} may need manual inspection (missing key sections)")

        except Exception as e:
            print(f"  Error verifying {os.path.basename(pdbqt_file)}: {str(e)}")

    def visualize_molecule(self, mol_file, format_type='pdb'):
        """Visualize molecule using py3Dmol"""
        try:
            with open(mol_file, 'r') as f:
                mol_content = f.read()

            view = py3Dmol.view(width=600, height=400)
            view.addModel(mol_content, format_type)
            view.setStyle({'stick': {}})
            view.zoomTo()
            view.show()

        except Exception as e:
            print(f"Visualization error for {os.path.basename(mol_file)}: {str(e)}")

    def create_summary_report(self, pdb_files, pdbqt_files):
        """Create a summary report of the conversion process"""
        report_filename = os.path.join(self.report_output_dir, "processing_report.md")

        report_content = f"""
# Ligand Processing Summary Report

## Input Information
- Number of input SMILES: {len(self.molecules)}
- Force field used: {self.selected_force_field} ({self.force_field_options.get(self.selected_force_field, 'Unknown')})
- OpenBabel path used: {self.obabel_path if self.obabel_path else 'Not found (manual conversion used)'}

## Output Files
- PDB files created: {len(pdb_files)} (located in `{os.path.basename(self.pdb_output_dir)}/`)
- PDBQT files created: {len(pdbqt_files)} (located in `{os.path.basename(self.pdbqt_output_dir)}/`)
- Ligands list: `ligands.txt` (located in `{os.path.basename(self.base_output_dir)}/`)

## File List
"""
        # Create a mapping from base filename to original SMILES/name
        mol_info = {mol.GetProp("_Name"): {"SMILES": mol.GetProp("SMILES"), "OriginalName": mol.GetProp("OriginalName")} for mol in self.molecules if mol.HasProp("_Name")}

        for i, pdbqt_file_path in enumerate(pdbqt_files, 1):
            base_name = os.path.basename(pdbqt_file_path).replace('.pdbqt', '')
            pdb_file_path = os.path.join(self.pdb_output_dir, f"{base_name}.pdb")

            original_name = mol_info.get(base_name, {}).get("OriginalName", f"ligand_{i}")
            smiles = mol_info.get(base_name, {}).get("SMILES", "Unknown")

            report_content += f"\n{i}. Ligand: {original_name}\n"
            report_content += f"  - SMILES: {smiles}\n"
            report_content += f"  - PDB: {os.path.basename(pdb_file_path)}\n"
            report_content += f"  - PDBQT: {os.path.basename(pdbqt_file_path)}\n"

        report_content += f"""
## Next Steps for AutoDock Vina
1. Your PDBQT files are ready for docking.
2. Prepare your protein receptor in PDBQT format (e.g., using `prepare_receptor4.py` from MGLTools).
3. Define the binding site search space (center and dimensions).
4. Run AutoDock Vina docking.

## Example Command for AutoDock Vina:
```bash
# Example docking command (adjust parameters as needed)
# Assuming 'protein.pdbqt' is your prepared receptor
# Replace X, Y, Z with your binding site coordinates
# Replace SIZE_X, SIZE_Y, SIZE_Z with your box dimensions
vina --receptor protein.pdbqt --ligand_list {os.path.join(os.path.basename(self.base_output_dir), 'ligands.txt')} \\
      --center_x 0 --center_y 0 --center_z 0 \\
      --size_x 20 --size_y 20 --size_z 20 \\
      --out docking_result.pdbqt --log log.txt
```

Report generated on: {pd.Timestamp.now()}
"""

        # Save report
        with open(report_filename, 'w') as f:
            f.write(report_content)

        print("\n" + "="*70)
        print("Summary Report:")
        print("="*70)
        print(report_content)
        print("="*70)
        print(f"Report saved to: {os.path.abspath(report_filename)}")
        return report_filename

    def create_ligands_list_file(self, pdbqt_files):
        """
        Creates a text file (ligands.txt) listing all generated PDBQT filenames.
        """
        list_filename = os.path.join(self.base_output_dir, "ligands.txt")
        print(f"\nCreating ligands list file: {list_filename}")

        with open(list_filename, 'w') as f:
            for pdbqt_file_path in pdbqt_files:
                f.write(os.path.basename(pdbqt_file_path) + '\n')

        print(f"✓ Ligands list file created: {list_filename}")
        return list_filename

    def create_results_zip(self, pdb_files, pdbqt_files, report_file_path, ligands_list_file_path):
        """
        Create a zip archive containing all result files (PDB, PDBQT, report, and ligands.txt).
        Files are organized into subfolders within the zip archive.
        """
        zip_filename = os.path.join(self.base_output_dir, "ligand_processing_results.zip")
        print(f"\nCreating zip archive: {zip_filename}")

        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            # Add all PDB files to 'PDB_files' folder inside zip
            for file_path in pdb_files:
                if os.path.exists(file_path):
                    arcname = os.path.join(os.path.basename(self.pdb_output_dir), os.path.basename(file_path))
                    zipf.write(file_path, arcname=arcname)
                    print(f"  Added: {arcname}")
                else:
                    print(f"  Warning: PDB file not found, skipping: {file_path}")

            # Add all PDBQT files to 'PDBQT_files' folder inside zip
            for file_path in pdbqt_files:
                if os.path.exists(file_path):
                    arcname = os.path.join(os.path.basename(self.pdbqt_output_dir), os.path.basename(file_path))
                    zipf.write(file_path, arcname=arcname)
                    print(f"  Added: {arcname}")
                else:
                    print(f"  Warning: PDBQT file not found, skipping: {file_path}")

            # Add the summary report to the root of the zip
            if os.path.exists(report_file_path):
                zipf.write(report_file_path, arcname=os.path.basename(report_file_path))
                print(f"  Added: {os.path.basename(report_file_path)}")
            else:
                print(f"  Warning: Report file not found, skipping: {report_file_path}")

            # Add the ligands.txt file to the root of the zip
            if os.path.exists(ligands_list_file_path):
                zipf.write(ligands_list_file_path, arcname=os.path.basename(ligands_list_file_path))
                print(f"  Added: {os.path.basename(ligands_list_file_path)}")
            else:
                print(f"  Warning: Ligands list file not found, skipping: {ligands_list_file_path}")

        print(f"✓ Zip archive created: {zip_filename}")
        return zip_filename

# Main execution function
def main():
    """Main function to run the ligand processing pipeline"""
    # Initialize processor with a base output directory, e.g., 'processed_ligands'
    # This will create 'processed_ligands/PDB_files' and 'processed_ligands/PDBQT_files'
    processor = LigandProcessor(base_output_dir="processed_ligands")

    print("SMILES to PDBQT Converter for AutoDock Vina")
    print("=" * 50)

    # Install dependencies
    try:
        processor.install_dependencies()
    except Exception as e:
        print(f"Warning: Some dependencies may not have installed correctly: {e}")
        print("Continuing with available tools...")

    # Input options
    print("\nInput Options:")
    print("1. Enter SMILES manually")
    print("2. Upload CSV file with SMILES")
    print("3. Upload text file with SMILES")
    print("4. Use example SMILES")

    choice = input("Select input method (1-4): ").strip()

    smiles_loaded_count = 0
    if choice == '1':
        smiles_input_str = input("Enter SMILES string(s) separated by commas: ")
        smiles_list_from_input = [s.strip() for s in smiles_input_str.split(',') if s.strip()]
        smiles_loaded_count = processor.load_smiles(smiles_list_from_input)

    elif choice == '2':
        print("\nCSV File Input Options:")
        print("1. File in Google Drive - enter relative path (e.g., 'data/molecules.csv')")
        print("2. File in current directory - enter filename only (e.g., 'molecules.csv')")
        print("3. Browse and upload file")

        csv_choice = input("Select option (1-3): ").strip()
        csv_path = None

        if csv_choice == '3' and files: # Check if google.colab.files is available
            print("Please upload your CSV file:")
            try:
                uploaded = files.upload()
                if uploaded:
                    csv_path = list(uploaded.keys())[0]
                    print(f"Uploaded file: {csv_path}")
                else:
                    print("No file uploaded. Reverting to example data.")
            except Exception as e:
                print(f"File upload failed: {e}. Reverting to example data.")
        elif csv_choice in ['1', '2']:
            csv_input_path = input("Enter path to CSV file: ")

            # Handle different path formats
            if not csv_input_path.startswith('/'):
                # Relative path - check in current working directory first
                full_path_cwd = os.path.join(work_dir, csv_input_path)
                if os.path.exists(full_path_cwd):
                    csv_path = full_path_cwd
                elif drive: # Try in Google Drive root if Drive is mounted
                    full_path_drive = os.path.join("/content/drive/MyDrive", csv_input_path)
                    if os.path.exists(full_path_drive):
                        csv_path = full_path_drive
            else: # Absolute path
                if os.path.exists(csv_input_path):
                    csv_path = csv_input_path

            if not csv_path:
                print(f"File not found: {csv_input_path}")
                print("Available files in current directory:")
                for file in os.listdir(work_dir):
                    if file.endswith('.csv'):
                        print(f"  - {file}")
                if drive and os.path.exists("/content/drive/MyDrive"):
                    print("Available files in Google Drive root:")
                    for file in os.listdir("/content/drive/MyDrive"):
                        if file.endswith('.csv'):
                            print(f"  - {file}")
                print("Reverting to example data.")

        if csv_path and os.path.exists(csv_path):
            smiles_loaded_count = processor.load_smiles(csv_path)
        else:
            print("No valid CSV path. Using example SMILES.")
            example_smiles = ["CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"] # Ibuprofen
            smiles_loaded_count = processor.load_smiles(example_smiles)

    elif choice == '3':
        txt_path = input("Enter path to text file: ")
        if os.path.exists(txt_path):
            smiles_loaded_count = processor.load_smiles(txt_path)
        else:
            print(f"Text file not found: {txt_path}. Using example SMILES.")
            example_smiles = ["CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"] # Ibuprofen
            smiles_loaded_count = processor.load_smiles(example_smiles)

    elif choice == '4':
        # Example SMILES for testing
        example_smiles = [
            "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",    # Ibuprofen
            "CC1=CC=C(C=C1)C(=O)NCCO",          # Acetaminophen derivative
            "C1=CC=C(C=C1)C(=O)O"               # Benzoic acid
        ]
        smiles_loaded_count = processor.load_smiles(example_smiles)
    else:
        print("Invalid choice. Using example SMILES.")
        example_smiles = ["CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"] # Ibuprofen
        smiles_loaded_count = processor.load_smiles(example_smiles)

    if smiles_loaded_count == 0:
        print("No valid molecules loaded. Exiting.")
        return

    # Process molecules
    pdb_files = processor.convert_to_pdb()

    if pdb_files:
        # Prepare for AutoDock Vina
        pdbqt_files = processor.prepare_autodock_vina(pdb_files)

        # Create ligands.txt file
        ligands_list_file_path = processor.create_ligands_list_file(pdbqt_files)

        # Create summary report
        report_file_path = processor.create_summary_report(pdb_files, pdbqt_files)

        # Visualize first molecule (if available)
        if pdb_files: # Visualize PDB, as PDBQT is not directly visualizable by py3Dmol
            print(f"\nVisualizing first molecule: {os.path.basename(pdb_files[0])}")
            try:
                if os.path.exists(pdb_files[0]):
                    processor.visualize_molecule(pdb_files[0])
                else:
                    print(f"Cannot visualize: PDB file '{pdb_files[0]}' not found.")
            except Exception as e:
                print(f"Visualization not available or failed: {e}")

        # --- Create zip file with all results and download (if in Colab) ---
        if pdbqt_files or pdb_files: # Only create zip if there are any generated files
            zip_file = processor.create_results_zip(pdb_files, pdbqt_files, report_file_path, ligands_list_file_path)

            # Provide download link in Colab
            if files: # Check if google.colab.files was successfully imported
                print("\nDownloading results zip file...")
                try:
                    files.download(zip_file)
                    print(f"✓ Download initiated for {zip_file}")
                except Exception as e:
                    print(f"Error initiating download: {e}")
            else:
                print("\nSkipping file download: Not running in Google Colab or 'google.colab.files' module not available.")
                print(f"Your results are saved in '{work_dir}' and the zip file is '{zip_file}'.")
        else:
            print("\nNo PDB or PDBQT files were generated, skipping zip creation.")

        print(f"\n✅ Processing complete!")
        print(f"📁 All result files saved in: {os.path.join(work_dir, processor.base_output_dir)}")
        print(f"📊 Summary report: {os.path.abspath(report_file_path)}")
        print(f"📄 Ligands list: {os.path.abspath(ligands_list_file_path)}")
        print(f"🧬 PDBQT files ready for AutoDock Vina: {len(pdbqt_files)}")

    else:
        print("❌ No PDB files were successfully created, so no PDBQT files or zip archive were generated.")

# Run the main function
if __name__ == "__main__":
    main()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 MB 40.3 MB/s eta 0:00:00
Mounting Google Drive...
Mounted at /content/drive
Working directory set to: /content/drive/MyDrive/Ligand_Processing
LigandProcessor initialized. Base output directory: processed_ligands
  PDB files will be saved in: processed_ligands/PDB_files
  PDBQT files will be saved in: processed_ligands/PDBQT_files
SMILES to PDBQT Converter for AutoDock Vina
Installing dependencies...
Installing OpenBabel via apt...
✓ OpenBabel installed via apt
✓ OpenBabel found at: /usr/bin/obabel
Dependency installation completed!

Input Options:
1. Enter SMILES manually
2. Upload CSV file with SMILES
3. Upload text file with SMILES
4. Use example SMILES
Select input method (1-4): 2

CSV File Input Options:
1. File in Google Drive - enter relative path (e.g., 'data/molecules.csv')
2. File in current directory - enter filename only (e.g., 'molecules.csv')
3. Browse and upload file
Select option (1-3): 3
Please upload your CSV file:

Saving valid_compounds_only.csv to valid_compounds_only.csv
Uploaded file: valid_compounds_only.csv
Loading file: valid_compounds_only.csv
CSV loaded successfully. Shape: (221, 8)
Available columns: ['cluster_id', 'compound_idx', 'Compound_Name', 'SMILES', 'cluster_size', 'selection_method', 'Status', 'Failure_Reason']
Auto-detected SMILES column: 'SMILES'
Found compound name column: 'Compound_Name'
Found 221 SMILES in column 'SMILES'
First 3 entries:
  1. Methotrexate: CN(CC1=CN=C2C(=N1)C(=NC(=N2)N)N)C3=CC=C(C=C3)C(=O)N[C@@H](CCC(=O)O)C(=O)O  
  2. NPC-61059: CC1(CCCC2(C13CC3C(=C)C=C2)C)C
  3. NPC-17446: CC(C)C(=O)OCC(C)C(C(C)(C)C)OC(=O)C(C)C
Successfully loaded 221 molecules out of 221 SMILES

Available Force Fields:
MMFF94: MMFF94 - Merck Molecular Force Field
MMFF94s: MMFF94s - MMFF94 static variant
UFF: UFF - Universal Force Field
ETKDG: ETKDG - Experimental-Torsion-angle Knowledge Distance Geometry
Select force field (MMFF94/MMFF94s/UFF/ETKDG): MMFF94
Selected force field: MMFF94

3Dmol.js failed to load for some reason. Please check your browser console for error messages.


Creating zip archive: processed_ligands/ligand_processing_results.zip
  Added: PDB_files/Methotrexate.pdb
  Added: PDB_files/NPC-61059.pdb
  Added: PDB_files/NPC-17446.pdb
  Added: PDB_files/NPC-38778.pdb
  Added: PDB_files/NPC-40664.pdb
  Added: PDB_files/NPC-644.pdb
  Added: PDB_files/NPC-6875.pdb
  Added: PDB_files/NPC-74333.pdb
  Added: PDB_files/MNC-6236.pdb
  Added: PDB_files/NPC-13511.pdb
  Added: PDB_files/NPC-56199.pdb
  Added: PDB_files/MNC-25741.pdb
  Added: PDB_files/NPC-64180.pdb
  Added: PDB_files/NPC-27230.pdb
  Added: PDB_files/MNC-23300.pdb
  Added: PDB_files/NPC-18330.pdb
  Added: PDB_files/NPC-46431.pdb
  Added: PDB_files/NPC-32153.pdb
  Added: PDB_files/NPC-9547.pdb
  Added: PDB_files/NPC-81419.pdb
  Added: PDB_files/MNC-13791.pdb
  Added: PDB_files/NPC-1706.pdb
  Added: PDB_files/NPC-33241.pdb
  Added: PDB_files/NPC-34142.pdb
  Added: PDB_files/NPC-7152.pdb
  Added: PDB_files/NPC-76708.pdb
  Added: PDB_files/NPC-55745.pdb
  Added: PDB_files/MNC-9089.pdb
  Added: P

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download initiated for processed_ligands/ligand_processing_results.zip

✅ Processing complete!
📁 All result files saved in: /content/drive/MyDrive/Ligand_Processing/processed_ligands
📊 Summary report: /content/drive/MyDrive/Ligand_Processing/processed_ligands/processing_report.md
📄 Ligands list: /content/drive/MyDrive/Ligand_Processing/processed_ligands/ligands.txt
🧬 PDBQT files ready for AutoDock Vina: 221


In [ ]:
#!/usr/bin/env python3
"""
Enhanced SMILES/Structure to PDBQT Converter
Supports: SMILES, PDB, MOL, MOL2 with automatic water removal
For Google Colab
"""

!pip install pandas numpy rdkit py3Dmol -q

import os, sys, subprocess, zipfile, glob, warnings
from pathlib import Path
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors, Descriptors
import py3Dmol
from IPython.display import display

try:
    from google.colab import drive, files
except ImportError:
    drive, files = None, None

warnings.filterwarnings('ignore')

# Google Drive Setup
work_dir = os.getcwd()
if drive:
    print("Mounting Google Drive...")
    try:
        drive.mount('/content/drive')
        work_dir = "/content/drive/MyDrive/Ligand_Processing"
        os.makedirs(work_dir, exist_ok=True)
        os.chdir(work_dir)
        print(f"✓ Working directory: {work_dir}")
    except Exception as e:
        print(f"⚠ Drive mount failed: {e}")


class EnhancedLigandProcessor:
    def __init__(self, base_output_dir="processed_ligands"):
        self.molecules = []
        self.original_names = []
        self.input_types = []
        self.force_field_options = {
            'MMFF94': 'MMFF94 - Merck Molecular Force Field',
            'MMFF94s': 'MMFF94s - MMFF94 static',
            'UFF': 'UFF - Universal Force Field',
            'ETKDG': 'ETKDG - Distance Geometry'
        }
        self.selected_force_field = None
        self.obabel_path = None

        self.base_output_dir = base_output_dir
        self.pdb_output_dir = os.path.join(base_output_dir, "PDB_files")
        self.pdbqt_output_dir = os.path.join(base_output_dir, "PDBQT_files")
        self.cleaned_input_dir = os.path.join(base_output_dir, "Cleaned_Input_Files")

        for d in [self.pdb_output_dir, self.pdbqt_output_dir, self.cleaned_input_dir]:
            os.makedirs(d, exist_ok=True)

        print(f"✓ Initialized: {base_output_dir}")

    def install_dependencies(self):
        """Install OpenBabel"""
        print("Installing dependencies...")
        try:
            subprocess.run(['apt-get', 'update', '-qq'], check=True, capture_output=True)
            subprocess.run(['apt-get', 'install', '-y', 'openbabel'], check=True, capture_output=True)
            print("✓ OpenBabel installed")
        except:
            try:
                subprocess.run(['pip', 'install', 'openbabel-wheel', '-q'], check=True)
                print("✓ OpenBabel installed via pip")
            except:
                print("⚠ OpenBabel not available - using manual conversion")

        for path in ['/usr/bin/obabel', '/opt/conda/bin/obabel', 'obabel']:
            try:
                if os.path.exists(path):
                    result = subprocess.run([path, '--help'], capture_output=True, timeout=5)
                    if result.returncode == 0:
                        self.obabel_path = path
                        print(f"✓ OpenBabel found: {path}")
                        break
            except:
                continue

    def remove_water(self, mol, name="molecule"):
        """Remove water molecules from structure"""
        if mol is None:
            return None
        try:
            frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
            non_water = []
            water_count = 0

            for frag in frags:
                formula = rdMolDescriptors.CalcMolFormula(frag)
                is_water = formula in ['H2O', 'OH2', 'HOH'] or (
                    frag.GetNumAtoms() <= 3 and
                    all(a.GetSymbol() in ['H', 'O'] for a in frag.GetAtoms())
                )

                if not is_water:
                    non_water.append(frag)
                else:
                    water_count += 1

            if water_count > 0:
                print(f"  Removed {water_count} water molecule(s)")

            return max(non_water, key=lambda x: x.GetNumAtoms()) if non_water else None
        except Exception as e:
            print(f"  Water removal error: {e}")
            return mol

    def sanitize_filename(self, name):
        """Create safe filename"""
        safe = str(name).replace(' ', '_').replace('/', '_').replace('\\', '_')
        return ''.join(c for c in safe if c.isalnum() or c in '_-.')[:100]

    def load_structure_file(self, file_path):
        """Load PDB/MOL/MOL2 file with water removal"""
        try:
            ext = os.path.splitext(file_path)[1].lower()
            base = os.path.splitext(os.path.basename(file_path))[0]

            print(f"\nProcessing {os.path.basename(file_path)}...")

            if ext == '.pdb':
                mol = Chem.MolFromPDBFile(file_path, removeHs=False, sanitize=True)
                file_type = 'PDB'
            elif ext in ['.mol', '.sdf']:
                mol = Chem.MolFromMolFile(file_path, removeHs=False, sanitize=True)
                file_type = 'MOL'
            elif ext == '.mol2':
                mol = Chem.MolFromMol2File(file_path, removeHs=False, sanitize=True)
                file_type = 'MOL2'
            else:
                print(f"  Unsupported format: {ext}")
                return None, None, None

            if mol is None:
                print(f"  Failed to load")
                return None, None, None

            print(f"  Loaded: {mol.GetNumAtoms()} atoms")

            mol_clean = self.remove_water(mol, base)
            if mol_clean:
                print(f"  Cleaned: {mol_clean.GetNumAtoms()} atoms")

                clean_path = os.path.join(self.cleaned_input_dir, f"{base}_cleaned.pdb")
                try:
                    Chem.MolToPDBFile(mol_clean, clean_path)
                    print(f"  Saved: {os.path.basename(clean_path)}")
                except Exception as e:
                    print(f"  Save warning: {e}")

            return mol_clean, base, file_type
        except Exception as e:
            print(f"  Error: {e}")
            return None, None, None

    def load_smiles(self, smiles_input):
        """Load SMILES from string/list/CSV/TXT"""
        self.molecules, self.original_names, self.input_types = [], [], []
        smiles_list = []

        if isinstance(smiles_input, str):
            if os.path.exists(smiles_input):
                if smiles_input.lower().endswith('.csv'):
                    df = pd.read_csv(smiles_input)
                    print(f"CSV loaded: {df.shape}")

                    smiles_col = None
                    for col in df.columns:
                        if col.lower() in ['smiles', 'smi', 'canonical_smiles']:
                            smiles_col = col
                            break

                    if not smiles_col:
                        print("Columns:", df.columns.tolist())
                        smiles_col = input("Enter SMILES column name: ").strip()

                    name_col = None
                    for col in df.columns:
                        if col.lower() in ['compound_name', 'name', 'id']:
                            name_col = col
                            break

                    smiles_list = df[smiles_col].dropna().astype(str).tolist()
                    self.original_names = (df[name_col].astype(str).tolist()[:len(smiles_list)]
                                          if name_col else
                                          [f"compound_{i+1}" for i in range(len(smiles_list))])

                elif smiles_input.lower().endswith(('.txt', '.smi')):
                    with open(smiles_input, 'r') as f:
                        smiles_list = [line.strip() for line in f if line.strip()]
                    self.original_names = [f"compound_{i+1}" for i in range(len(smiles_list))]
                else:
                    smiles_list = [smiles_input]
                    self.original_names = ["ligand_1"]
        elif isinstance(smiles_input, list):
            smiles_list = smiles_input
            self.original_names = [f"ligand_{i+1}" for i in range(len(smiles_list))]

        success = 0
        for i, smi in enumerate(smiles_list):
            try:
                mol = Chem.MolFromSmiles(str(smi).strip())
                if mol:
                    name = self.sanitize_filename(self.original_names[i])
                    mol.SetProp("_Name", name)
                    mol.SetProp("SMILES", str(smi).strip())
                    mol.SetProp("OriginalName", str(self.original_names[i]))
                    self.molecules.append(mol)
                    self.input_types.append('SMILES')
                    success += 1
            except Exception as e:
                print(f"SMILES {i+1} error: {e}")

        print(f"✓ Loaded {success}/{len(smiles_list)} SMILES")
        return success

    def load_structure_files(self, paths_or_dir):
        """Load multiple structure files"""
        files = []
        if isinstance(paths_or_dir, str):
            if os.path.isdir(paths_or_dir):
                for ext in ['*.pdb', '*.mol', '*.mol2', '*.sdf']:
                    files.extend(glob.glob(os.path.join(paths_or_dir, ext)))
            elif os.path.isfile(paths_or_dir):
                files = [paths_or_dir]
        elif isinstance(paths_or_dir, list):
            files = paths_or_dir

        print(f"Found {len(files)} structure file(s)")

        success = 0
        for path in files:
            mol, base, ftype = self.load_structure_file(path)
            if mol:
                name = self.sanitize_filename(base)
                mol.SetProp("_Name", name)
                mol.SetProp("OriginalName", base)
                try:
                    mol.SetProp("SMILES", Chem.MolToSmiles(mol))
                except:
                    mol.SetProp("SMILES", "N/A")

                self.molecules.append(mol)
                self.original_names.append(base)
                self.input_types.append(ftype)
                success += 1

        print(f"✓ Loaded {success}/{len(files)} structures")
        return success

    def select_force_field(self):
        """Select force field for 3D generation"""
        print("\nForce Fields:")
        for k, v in self.force_field_options.items():
            print(f"  {k}: {v}")

        while True:
            choice = input("Select (MMFF94/MMFF94s/UFF/ETKDG): ").strip().upper()
            if choice in self.force_field_options:
                self.selected_force_field = choice
                print(f"✓ Selected: {choice}")
                break

    def generate_3d(self, mol, ff):
        """Generate 3D conformer"""
        mol_h = Chem.AddHs(mol)

        if ff == 'ETKDG':
            params = AllChem.ETKDGv3()
            params.randomSeed = 42
            AllChem.EmbedMolecule(mol_h, params)
            try:
                AllChem.MMFFOptimizeMolecule(mol_h)
            except:
                pass
        else:
            AllChem.EmbedMolecule(mol_h, randomSeed=42)
            try:
                if ff == 'MMFF94':
                    AllChem.MMFFOptimizeMolecule(mol_h)
                elif ff == 'MMFF94s':
                    AllChem.MMFFOptimizeMolecule(mol_h, mmffVariant='MMFF94s')
                elif ff == 'UFF':
                    AllChem.UFFOptimizeMolecule(mol_h)
            except:
                pass

        return mol_h

    def convert_to_pdb(self):
        """Convert to PDB format"""
        if not self.selected_force_field:
            self.select_force_field()

        pdb_files = []
        print(f"\nGenerating 3D with {self.selected_force_field}...")

        for i, mol in enumerate(self.molecules):
            try:
                input_type = self.input_types[i] if i < len(self.input_types) else 'UNKNOWN'

                if input_type in ['PDB', 'MOL2', 'MOL'] and mol.GetNumConformers() > 0:
                    print(f"  {mol.GetProp('_Name')}: Using existing 3D")
                    mol_3d = Chem.AddHs(mol)
                else:
                    print(f"  {mol.GetProp('_Name')}: Generating 3D")
                    mol_3d = self.generate_3d(mol, self.selected_force_field)

                name = mol.GetProp("_Name")
                pdb_file = os.path.join(self.pdb_output_dir, f"{name}.pdb")

                Chem.MolToPDBFile(mol_3d, pdb_file)
                pdb_files.append(pdb_file)
                print(f"    ✓ {os.path.basename(pdb_file)}")
            except Exception as e:
                print(f"    ✗ Error: {e}")

        print(f"\n✓ Created {len(pdb_files)} PDB files")
        return pdb_files

    def prepare_vina(self, pdb_files):
        """Convert to PDBQT for AutoDock Vina"""
        print("\nPreparing PDBQT files...")
        pdbqt_files = []

        for pdb_file in pdb_files:
            try:
                mol = Chem.MolFromPDBFile(pdb_file, removeHs=False)
                if not mol:
                    continue

                print(f"\n  {os.path.basename(pdb_file)}")

                AllChem.ComputeGasteigerCharges(mol)
                num_rot = rdMolDescriptors.CalcNumRotatableBonds(mol)
                print(f"    Rotatable bonds: {num_rot}")

                name = os.path.basename(pdb_file).replace('.pdb', '')
                pdbqt_file = os.path.join(self.pdbqt_output_dir, f"{name}.pdbqt")

                if self.obabel_path:
                    try:
                        cmd = f"{self.obabel_path} {pdb_file} -O {pdbqt_file} --addpolarh --partialcharge gasteiger"
                        subprocess.run(cmd, shell=True, check=True, capture_output=True)
                        if os.path.exists(pdbqt_file) and os.path.getsize(pdbqt_file) > 0:
                            print(f"    ✓ OpenBabel")
                            pdbqt_files.append(pdbqt_file)
                            continue
                    except:
                        pass

                if self.manual_pdbqt(pdb_file, pdbqt_file, num_rot, mol):
                    print(f"    ✓ Manual conversion")
                    pdbqt_files.append(pdbqt_file)

            except Exception as e:
                print(f"    ✗ {e}")

        print(f"\n✓ Created {len(pdbqt_files)} PDBQT files")
        return pdbqt_files

    def manual_pdbqt(self, pdb_file, out_file, num_rot, mol):
        """Manual PDBQT conversion"""
        try:
            with open(pdb_file) as f:
                lines = f.readlines()

            pdbqt = []
            name = mol.GetProp("_Name") if mol.HasProp("_Name") else "ligand"

            pdbqt.extend([
                f"REMARK  Name = {name}\n",
                f"REMARK  {num_rot} active torsions:\n",
                "ROOT\n"
            ])

            atom_idx = 0
            for line in lines:
                if line.startswith(('ATOM', 'HETATM')):
                    element = line[76:78].strip()
                    ad_type = {'C':'C','N':'N','O':'O','S':'S','P':'P','F':'F',
                              'Cl':'Cl','Br':'Br','I':'I','H':'H'}.get(element, element)

                    charge = 0.0
                    if atom_idx < mol.GetNumAtoms():
                        atom = mol.GetAtomWithIdx(atom_idx)
                        if atom.HasProp('_GasteigerCharge'):
                            try:
                                charge = float(atom.GetProp('_GasteigerCharge'))
                            except:
                                pass
                    atom_idx += 1

                    pdbqt_line = (
                        f"{line[:66]}"
                        f"{charge:+6.3f} {ad_type:<2}\n"
                    )
                    pdbqt.append(pdbqt_line)

            pdbqt.extend([
                "ENDROOT\n",
                f"TORSDOF {num_rot}\n"
            ])

            with open(out_file, 'w') as f:
                f.writelines(pdbqt)

            return os.path.exists(out_file)
        except:
            return False

    def create_ligands_list(self, pdbqt_files):
        """Create ligands.txt"""
        list_file = os.path.join(self.base_output_dir, "ligands.txt")
        with open(list_file, 'w') as f:
            for p in pdbqt_files:
                f.write(os.path.basename(p) + '\n')
        print(f"✓ Created: ligands.txt")
        return list_file

    def create_report(self, pdb_files, pdbqt_files):
        """Generate summary report"""
        report_file = os.path.join(self.base_output_dir, "processing_report.md")

        report = f"""# Ligand Processing Report

## Summary
- Molecules processed: {len(self.molecules)}
- Input types: {', '.join(set(self.input_types))}
- Force field: {self.selected_force_field}
- PDB files: {len(pdb_files)}
- PDBQT files: {len(pdbqt_files)}

## Files
"""
        for i, pdbqt in enumerate(pdbqt_files, 1):
            name = os.path.basename(pdbqt).replace('.pdbqt', '')
            report += f"\n{i}. {name}\n"
            report += f"   - PDB: {name}.pdb\n"
            report += f"   - PDBQT: {name}.pdbqt\n"

        report += f"""
## AutoDock Vina Usage
```bash
vina --receptor protein.pdbqt \\
     --ligand_list ligands.txt \\
     --center_x 0 --center_y 0 --center_z 0 \\
     --size_x 20 --size_y 20 --size_z 20 \\
     --out results.pdbqt
```
"""

        with open(report_file, 'w') as f:
            f.write(report)

        print(f"\n✓ Report saved: {os.path.basename(report_file)}")
        return report_file

    def create_zip(self, pdb_files, pdbqt_files, report, ligands_list):
        """Create results ZIP"""
        zip_file = os.path.join(self.base_output_dir, "results.zip")

        with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
            for p in pdb_files:
                if os.path.exists(p):
                    zf.write(p, os.path.join("PDB_files", os.path.basename(p)))

            for p in pdbqt_files:
                if os.path.exists(p):
                    zf.write(p, os.path.join("PDBQT_files", os.path.basename(p)))

            if os.path.exists(report):
                zf.write(report, os.path.basename(report))

            if os.path.exists(ligands_list):
                zf.write(ligands_list, os.path.basename(ligands_list))

            # Add cleaned files
            for f in os.listdir(self.cleaned_input_dir):
                fpath = os.path.join(self.cleaned_input_dir, f)
                zf.write(fpath, os.path.join("Cleaned_Input_Files", f))

        print(f"✓ ZIP created: {os.path.basename(zip_file)}")
        return zip_file

    def visualize(self, pdb_file):
        """Visualize molecule"""
        try:
            with open(pdb_file) as f:
                view = py3Dmol.view(width=600, height=400)
                view.addModel(f.read(), 'pdb')
                view.setStyle({'stick': {}})
                view.zoomTo()
                view.show()
        except Exception as e:
            print(f"Visualization error: {e}")


def main():
    """Main execution"""
    processor = EnhancedLigandProcessor()

    print("="*60)
    print("ENHANCED LIGAND PROCESSOR")
    print("Supports: SMILES, PDB, MOL, MOL2 + Water Removal")
    print("="*60)

    processor.install_dependencies()

    print("\nInput Options:")
    print("1. Enter SMILES manually")
    print("2. Upload CSV with SMILES")
    print("3. Upload structure files (PDB/MOL/MOL2)")
    print("4. Directory with structure files")
    print("5. Use example data")

    choice = input("\nSelect (1-5): ").strip()

    loaded = 0

    if choice == '1':
        smiles = input("Enter SMILES (comma-separated): ")
        loaded = processor.load_smiles([s.strip() for s in smiles.split(',') if s.strip()])

    elif choice == '2':
        if files:
            print("Upload CSV file:")
            uploaded = files.upload()
            if uploaded:
                loaded = processor.load_smiles(list(uploaded.keys())[0])
        else:
            path = input("Enter CSV path: ")
            if os.path.exists(path):
                loaded = processor.load_smiles(path)

    elif choice == '3':
        if files:
            print("Upload structure files (PDB/MOL/MOL2):")
            uploaded = files.upload()
            if uploaded:
                loaded = processor.load_structure_files(list(uploaded.keys()))
        else:
            path = input("Enter file path: ")
            if os.path.exists(path):
                loaded = processor.load_structure_files(path)

    elif choice == '4':
        dir_path = input("Enter directory path: ")
        if os.path.isdir(dir_path):
            loaded = processor.load_structure_files(dir_path)

    else:  # Example
        example = ["CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"]  # Ibuprofen
        loaded = processor.load_smiles(example)

    if loaded == 0:
        print("\n❌ No molecules loaded. Exiting.")
        return

    # Process
    pdb_files = processor.convert_to_pdb()

    if pdb_files:
        pdbqt_files = processor.prepare_vina(pdb_files)
        ligands_list = processor.create_ligands_list(pdbqt_files)
        report = processor.create_report(pdb_files, pdbqt_files)

        # Visualize first molecule
        if pdb_files:
            print(f"\nVisualizing: {os.path.basename(pdb_files[0])}")
            processor.visualize(pdb_files[0])

        # Create ZIP and download
        zip_file = processor.create_zip(pdb_files, pdbqt_files, report, ligands_list)

        if files:
            print("\nDownloading results...")
            files.download(zip_file)

        print(f"\n{'='*60}")
        print("✅ PROCESSING COMPLETE!")
        print(f"{'='*60}")
        print(f"📁 Output: {processor.base_output_dir}")
        print(f"📊 Report: processing_report.md")
        print(f"📄 Ligands: ligands.txt")
        print(f"🧬 PDBQT files: {len(pdbqt_files)}")
        print(f"💧 Cleaned files: {len(os.listdir(processor.cleaned_input_dir))}")
    else:
        print("\n❌ No PDB files created")


if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 24.9 MB/s eta 0:00:00
Mounting Google Drive...
Mounted at /content/drive
✓ Working directory: /content/drive/MyDrive/Ligand_Processing
✓ Initialized: processed_ligands
ENHANCED LIGAND PROCESSOR
Supports: SMILES, PDB, MOL, MOL2 + Water Removal
Installing dependencies...
✓ OpenBabel installed
✓ OpenBabel found: /usr/bin/obabel

Input Options:
1. Enter SMILES manually
2. Upload CSV with SMILES
3. Upload structure files (PDB/MOL/MOL2)
4. Directory with structure files
5. Use example data

Select (1-5): 2
Upload CSV file:


Saving generated_decoys.csv to generated_decoys.csv
CSV loaded: (500, 2)
✓ Loaded 500/500 SMILES

Force Fields:
  MMFF94: MMFF94 - Merck Molecular Force Field
  MMFF94s: MMFF94s - MMFF94 static
  UFF: UFF - Universal Force Field
  ETKDG: ETKDG - Distance Geometry
Select (MMFF94/MMFF94s/UFF/ETKDG): MMFF94
✓ Selected: MMFF94

Generating 3D with MMFF94...
  DPC-17_1: Generating 3D
    ✓ DPC-17_1.pdb
  DPC-17_2: Generating 3D
    ✓ DPC-17_2.pdb
  DPC-17_3: Generating 3D
    ✓ DPC-17_3.pdb
  DPC-17_4: Generating 3D
    ✓ DPC-17_4.pdb
  DPC-17_5: Generating 3D
    ✓ DPC-17_5.pdb
  DPC-17_6: Generating 3D
    ✓ DPC-17_6.pdb
  DPC-17_7: Generating 3D
    ✓ DPC-17_7.pdb
  DPC-17_8: Generating 3D
    ✓ DPC-17_8.pdb
  DPC-17_9: Generating 3D
    ✓ DPC-17_9.pdb
  DPC-17_10: Generating 3D
    ✓ DPC-17_10.pdb
  DPC-17_11: Generating 3D
    ✓ DPC-17_11.pdb
  DPC-17_12: Generating 3D
    ✓ DPC-17_12.pdb
  DPC-17_13: Generating 3D
    ✓ DPC-17_13.pdb
  DPC-17_14: Generating 3D
    ✓ DPC-17_14.pdb
  DPC

[15:18:27] UFFTYPER: Unrecognized atom type: Se2+2 (10)


    ✓ DPC-23_12.pdb
  DPC-23_13: Generating 3D
    ✓ DPC-23_13.pdb
  DPC-23_14: Generating 3D
    ✓ DPC-23_14.pdb
  DPC-23_15: Generating 3D
    ✓ DPC-23_15.pdb
  DPC-23_16: Generating 3D
    ✓ DPC-23_16.pdb
  DPC-23_17: Generating 3D
    ✓ DPC-23_17.pdb
  DPC-23_18: Generating 3D
    ✓ DPC-23_18.pdb
  DPC-23_19: Generating 3D
    ✓ DPC-23_19.pdb
  DPC-23_20: Generating 3D
    ✓ DPC-23_20.pdb
  DPC-23_21: Generating 3D
    ✓ DPC-23_21.pdb
  DPC-23_22: Generating 3D
    ✓ DPC-23_22.pdb
  DPC-23_23: Generating 3D
    ✓ DPC-23_23.pdb
  DPC-23_24: Generating 3D
    ✓ DPC-23_24.pdb
  DPC-23_25: Generating 3D
    ✓ DPC-23_25.pdb
  DPC-23_26: Generating 3D
    ✓ DPC-23_26.pdb
  DPC-23_27: Generating 3D
    ✓ DPC-23_27.pdb
  DPC-23_28: Generating 3D
    ✓ DPC-23_28.pdb
  DPC-23_29: Generating 3D
    ✓ DPC-23_29.pdb
  DPC-23_30: Generating 3D
    ✓ DPC-23_30.pdb
  DPC-23_31: Generating 3D
    ✓ DPC-23_31.pdb
  DPC-23_32: Generating 3D
    ✓ DPC-23_32.pdb
  DPC-23_33: Generating 3D
    ✓ DPC-23_

[15:18:36] UFFTYPER: Unrecognized charge state for atom: 1


...

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

✓ ZIP created: results.zip



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ PROCESSING COMPLETE!
📁 Output: processed_ligands
📊 Report: processing_report.md
📄 Ligands: ligands.txt
🧬 PDBQT files: 500
💧 Cleaned files: 0
